# 12 — S.E.T.S Score

## Goal
Combine the four individual scores into a single **S.E.T.S** number and rate each zone as A, B, or skip.

---

## The Five Components

| Letter | Concept | Notebook | Max |
|--------|---------|----------|-----|
| **S** | Strength (departure ATR-ratio) | NB06 | 2 |
| **T** | Time (base candle count)       | NB09 | 2 |
| **F** | Freshness (touch count)        | NB08 | 2 |
| **A** | Alignment (trend)              | NB11 | 2 |
| **C** | Curve (HTF position)           | NB10 | 2 |

$$\text{SETS} = S + T + F + A + C \quad \in [0, 10]$$

---

## Strength Mapping (departure ATR-ratio → score)

$$S = \begin{cases} 2 & \text{dep\_atr} \geq 3.0 \\ 1 & \text{dep\_atr} \geq 1.5 \\ 0 & \text{otherwise} \end{cases}$$

---

## Rating

| SETS total | Rating |
|------------|--------|
| ≥ 7        | ★★★ A-setup — take it |
| 5 – 6      | ★★ B-setup — trade with caution |
| ≤ 4        | ★ Skip |

---

## Visualization
Horizontal rating bands on a bar chart of SETS scores per zone.  
Only ★★★ and ★★ zones are overlaid on the candle chart.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. Compute all sub-scores and combine into SETS

In [ ]:
# ── helpers from NB08–11 ────────────────────────────────────────────────────
SWING_WINDOW = 3
FRESHNESS_SCORE_MAP = {0: 2, 1: 1}
STRENGTH_SCORE = lambda da: 2 if da >= 3.0 else (1 if da >= 1.5 else 0)

htf_high  = h_arr.max(); htf_low = l_arr.min(); htf_range = htf_high - htf_low

def count_touches(zone):
    prox, dist, zt = zone["proximal"], zone["distal"], zone["zone_type"]
    start  = zone["be"] + DEPARTURE_CANDLES + 1
    touches = 0
    for i in range(start, len(c_arr)):
        close = c_arr[i]
        if zt == "demand" and close < dist: break
        if zt == "supply" and close > dist: break
        if zt == "demand" and dist <= close <= prox: touches += 1
        if zt == "supply" and prox <= close <= dist: touches += 1
    return touches

def time_score(zone):
    n = zone["be"] - zone["bs"] + 1
    return 2 if n <= 2 else (1 if n == 3 else 0)

def curve_score(zone):
    pos = (zone["proximal"] - htf_low) / htf_range if htf_range > 0 else 0.5
    third = "low" if pos < 0.333 else ("high" if pos > 0.667 else "mid")
    return ({"low": 2, "mid": 1, "high": 0} if zone["zone_type"] == "demand"
            else {"low": 0, "mid": 1, "high": 2})[third]

def find_swings():
    sh_idx, sl_idx = [], []
    for i in range(SWING_WINDOW, len(h_arr) - SWING_WINDOW):
        if h_arr[i] == h_arr[i-SWING_WINDOW:i+SWING_WINDOW+1].max(): sh_idx.append(i)
        if l_arr[i] == l_arr[i-SWING_WINDOW:i+SWING_WINDOW+1].min(): sl_idx.append(i)
    return sh_idx, sl_idx

sh_idx, sl_idx = find_swings()
TREND_SCORE_MAP = {
    ("demand","uptrend"):2, ("demand","sideways"):1, ("demand","downtrend"):0,
    ("supply","downtrend"):2, ("supply","sideways"):1, ("supply","uptrend"):0,
}

def trend_at(pos):
    past_sh = [i for i in sh_idx if i < pos]
    past_sl = [i for i in sl_idx if i < pos]
    if len(past_sh) < 2 or len(past_sl) < 2: return "sideways"
    hh = h_arr[past_sh[-1]] > h_arr[past_sh[-2]]; hl = l_arr[past_sl[-1]] > l_arr[past_sl[-2]]
    lh = h_arr[past_sh[-1]] < h_arr[past_sh[-2]]; ll = l_arr[past_sl[-1]] < l_arr[past_sl[-2]]
    if hh and hl: return "uptrend"
    if lh and ll: return "downtrend"
    return "sideways"

def sets_rating(total):
    return "★★★ A" if total >= 7 else ("★★ B" if total >= 5 else "★ Skip")

for z in zones:
    t = count_touches(z)
    z["s_strength"]  = STRENGTH_SCORE(z["dep_atr"])
    z["s_time"]      = time_score(z)
    z["s_freshness"] = FRESHNESS_SCORE_MAP.get(t, 0)
    z["s_curve"]     = curve_score(z)
    z["s_align"]     = TREND_SCORE_MAP.get((z["zone_type"], trend_at(z["bs"])), 1)
    z["sets_total"]  = z["s_strength"] + z["s_time"] + z["s_freshness"] + z["s_curve"] + z["s_align"]
    z["rating"]      = sets_rating(z["sets_total"])
    z["scenario"]    = labeled["scenario"].iloc[z["bs"]]

print(f"{'Scenario':<22} {'Type':8} {'S':>2} {'T':>2} {'F':>2} {'A':>2} {'C':>2} {'Total':>5} {'Rating'}")
print("-" * 65)
for z in zones:
    print(f"{z['scenario']:<22} {z['zone_type']:8} "
          f"{z['s_strength']:>2} {z['s_time']:>2} {z['s_freshness']:>2} "
          f"{z['s_align']:>2} {z['s_curve']:>2} {z['sets_total']:>5}  {z['rating']}")


## 3. Visualization — SETS bar chart + A/B zones on price

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

RATING_COLOR = {"★★★ A": "#26a69a", "★★ B": "#ffb74d", "★ Skip": "#ef5350"}

fig = make_subplots(rows=2, cols=1, shared_xaxes=False, row_heights=[0.6, 0.4],
                    subplot_titles=["Price — A/B zones highlighted", "SETS total per zone"],
                    vertical_spacing=0.1)

fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
), row=1, col=1)

for z in zones:
    if z["rating"] == "★ Skip": continue
    x0, x1 = df.index[z["bs"]], df.index[z["be"]]
    c_edge  = RATING_COLOR[z["rating"]]
    alpha   = 0.20 if z["rating"] == "★★★ A" else 0.10
    fill    = f"rgba(38,166,154,{alpha})" if z["zone_type"] == "demand" else f"rgba(239,83,80,{alpha})"
    fig.add_vrect(x0=x0, x1=x1, fillcolor=fill,
                  line=dict(color=c_edge, width=1.5), row=1, col=1)
    fig.add_annotation(x=x0, y=z["proximal"],
                       text=f"{z['formation']} {z['rating']}",
                       showarrow=False, font=dict(size=8, color=c_edge),
                       xanchor="left", yanchor="bottom")

labels   = [z["scenario"] for z in zones]
totals   = [z["sets_total"] for z in zones]
bar_cols = [RATING_COLOR[z["rating"]] for z in zones]

fig.add_trace(go.Bar(x=labels, y=totals, marker_color=bar_cols, name="SETS total"), row=2, col=1)
fig.add_hline(y=7, line=dict(color="#26a69a", dash="dash"), row=2, col=1)
fig.add_hline(y=5, line=dict(color="#ffb74d", dash="dash"), row=2, col=1)

fig.update_layout(
    title="S.E.T.S Scoring — zone ratings",
    height=660, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    showlegend=False,
)
for ax in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
    fig.update_layout(**{ax: dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)})
fig.show()
